# Actividad 2 · Bajar la montaña: minimización vs. optimización
### Descenso de gradiente sobre el paisaje de Lennard-Jones
**Taller:** Física y matemática detrás de los campos de fuerza y la dinámica molecular · XXXIV Semana de la Química, Bogotá 2026

---

**Duración:** ~30 min · **Herramienta:** Google Colab o Jupyter · **Prerrequisito:** Actividad 1 (el potencial y la fuerza de Lennard-Jones)

**Objetivos:**
1. Entender qué es un paisaje energético y qué significa "minimizar".
2. Implementar el algoritmo de descenso de gradiente desde cero.
3. Descubrir experimentalmente por qué el tamaño del paso importa.
4. Distinguir minimización local (MMFF94) de optimización global (búsqueda de poses) y conectar ambas con el *docking*.

## Concepto clave

En la Actividad 1 se observó que el campo de fuerza define un **paisaje energético**: una montaña donde la altura es la energía $U$. Un sistema físico tiende a alcanzar el valle de menor energía accesible. El algoritmo que lo lleva allí es el **descenso de gradiente**:

$$r_{n+1} = r_n + \alpha\, F(r_n) = r_n - \alpha\,\frac{dU}{dr}\Big|_{r_n}$$

- $\alpha$ (**paso** o *learning rate*): qué tanto avanzamos en cada iteración.
- $F = -dU/dr$: la fuerza siempre apunta "cuesta abajo" (derivada en la Actividad 1).

> **Distinción:** el descenso de gradiente actualiza las coordenadas con un tamaño de paso $\alpha$ y puede converger a un mínimo local. El docking realiza una búsqueda heurística y aproximada sobre poses y conformaciones; no garantiza encontrar el mínimo global. Ambos problemas se relacionan con una función de energía o puntuación, pero no son el mismo procedimiento.

## Paso 0 · Preparativos

La celda siguiente reutiliza el potencial y la fuerza de la Actividad 1. Ejecutar la celda.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def U_lennard_jones(r, epsilon=1.0, sigma=1.0):
    """Potencial de Lennard-Jones U(r)."""
    return 4 * epsilon * ((sigma / r)**12 - (sigma / r)**6)

def F_lennard_jones(r, epsilon=1.0, sigma=1.0):
    """Fuerza F(r) = -dU/dr."""
    return (24 * epsilon / r) * (2 * (sigma / r)**12 - (sigma / r)**6)

## Experimento 1 · Minimización: rodar hasta el valle más próximo

Se parte de un átomo en $r_0 = 1.6\,\sigma$ (a la derecha del mínimo) y se aplica la fuerza para desplazarlo cuesta abajo. Completar el paso de actualización.

**Predicción previa:** ¿Cuántos pasos se estiman necesarios? ¿A qué valor de $r$ se llegará? (El mínimo de la Actividad 1 está en $r = 2^{1/6}\sigma \approx 1.122$.)

In [ ]:
# --- Parámetros del descenso ---
r0    = 1.6     # posición inicial (a la derecha del mínimo)
alpha = 0.01    # tamaño del paso
tol   = 1e-8    # criterio de parada: cuando |F| sea prácticamente cero

trayectoria = [r0]          # guardamos todas las posiciones visitadas
r = r0
for paso in range(10000):
    fuerza = F_lennard_jones(r)
    # Solución: actualiza r usando el descenso de gradiente
    #   r = r + ...?
    r = r + alpha * fuerza
    trayectoria.append(r)
    if abs(fuerza) < tol:
        break

print(f"Convergió en {len(trayectoria)-1} pasos")
print(f"r final = {r:.6f}   (mínimo teórico: 2^(1/6) = {2**(1/6):.6f})")

# Gráfica: el átomo "rodando" cuesta abajo en el paisaje
rr = np.linspace(0.9, 3.0, 400)
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(rr, U_lennard_jones(rr), color='gray', lw=1, label='paisaje U(r)')
ax.plot(trayectoria, U_lennard_jones(np.array(trayectoria)), 'o-', color='darkred',
        ms=4, lw=1, label='descenso de gradiente')
ax.annotate('punto inicial', xy=(r0, U_lennard_jones(r0)), xytext=(2.1, -0.2),
            arrowprops=dict(arrowstyle='->', color='black'))
ax.annotate('mínimo alcanzado', xy=(r, U_lennard_jones(r)), xytext=(0.95, -0.75),
            arrowprops=dict(arrowstyle='->', color='black'))
ax.set_xlabel('r'); ax.set_ylabel('U(r)')
ax.set_title('Minimización: rodar hasta el valle más próximo')
ax.legend(); ax.grid(alpha=0.3)
plt.show()

## Experimento 2 · El tamaño del paso: ¿por qué importa $\alpha$?

Con la misma meta y distintos valores del paso. Ejecutar la celda para tres valores de $\alpha$ y analizar los resultados:

- **$\alpha$ pequeño** (0.001): converge, pero lento.
- **$\alpha$ adecuado** (0.01): converge rápido y estable.
- **$\alpha$ grande** (0.2): ¡inestable! El átomo "salta" por encima del valle y la energía se dispara.

> **Nota:** $\alpha$ controla la magnitud de cada actualización del descenso de gradiente. En AutoDock Vina, `exhaustiveness` regula el esfuerzo computacional dedicado a la búsqueda de poses. Son parámetros distintos: aumentar el esfuerzo de búsqueda puede mejorar la exploración, pero no garantiza hallar el mínimo global.

**Pregunta:** ¿Es posible estimar aproximadamente un valor crítico de $\alpha$ a partir de la curvatura del pozo?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2), sharey=True)
r_min_teorico = 2**(1/6)
for ax, alpha in zip(axes, [0.001, 0.01, 0.2]):
    r = 1.6
    tray = [r]
    for paso in range(2000):
        r = r + alpha * F_lennard_jones(r)
        tray.append(r)
        if not np.isfinite(r) or r < 0.5 or abs(r - r_min_teorico) < 1e-5:
            break
    tray = np.array(tray)
    convergio = np.isfinite(r) and abs(r - r_min_teorico) < 0.01
    titulo = f'$\\alpha$ = {alpha}: ' + ('converge' if convergio else '¡inestable!')
    ax.plot(tray, 'o-', ms=3, lw=1, color='darkred' if convergio else 'black')
    ax.axhline(r_min_teorico, color='green', ls='--', lw=1, label='mínimo teórico')
    ax.set_title(titulo)
    ax.set_xlabel('paso'); ax.grid(alpha=0.3)
    ax.set_ylim(0.5, 3.0)
    ax.legend(fontsize=8)
axes[0].set_ylabel('r')
plt.suptitle('Misma meta, distinto paso', y=1.03)
plt.tight_layout()
plt.show()

print("Lectura de la gráfica:")
print("  alpha = 0.001 -> converge despacio (cientos de pasos)")
print("  alpha = 0.01  -> converge rápido y estable (~55 pasos)")
print("  alpha = 0.2   -> salta por encima del pozo y se pierde (linea plana fuera de escala)")

## Experimento 3 · De un valle a una cordillera: local vs. global

El mundo real no tiene un solo valle. Se considera un paisaje con **dos pozos** (un potencial de doble pozo, un modelo simplificado de un sitio de unión con dos sub-bolsillos):

$$U(x) = x^4 - 2x^2 + 0.2\,x, \qquad F(x) = -\frac{dU}{dx} = -(4x^3 - 4x + 0.2)$$

El término $0.2\,x$ rompe la simetría: un pozo es más profundo que el otro. Ejecutar la celda y determinar a qué mínimo converge cada punto inicial.

In [ ]:
def U_doble(x):
    return x**4 - 2*x**2 + 0.2*x

def F_doble(x):
    return -(4*x**3 - 4*x + 0.2)   # fuerza = -dU/dx

puntos_iniciales = [-1.5, -0.1, 0.1, 1.5]
resultados = []
for x0 in puntos_iniciales:
    x = x0
    for _ in range(5000):
        x = x + 0.05 * F_doble(x)
    resultados.append((x0, x, U_doble(x)))
    print(f"x0 = {x0:+.1f}  ->  mínimo en x = {x:+.4f}   U = {U_doble(x):+.4f}")

# Gráfica del paisaje con las trayectorias de cada inicio
xx = np.linspace(-1.8, 1.8, 400)
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(xx, U_doble(xx), color='gray', lw=1.5, label='paisaje U(x)')
colores = ['navy', 'teal', 'orange', 'darkred']
for (x0, xf, uf), c in zip(resultados, colores):
    ax.plot([x0], [U_doble(x0)], 'o', color=c, ms=8)
    ax.plot([xf], [uf], 's', color=c, ms=8,
            label=f'inicio {x0:+.1f} → mínimo {xf:+.2f}')
ax.set_xlabel('x'); ax.set_ylabel('U(x)')
ax.set_title('Distintos inicios, distintos valles')
ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.show()

ganador = min(resultados, key=lambda t: t[2])
print(f"\n Mejor mínimo global: x = {ganador[1]:.4f} con U = {ganador[2]:.4f} "
      f"(lo alcanzó el punto inicial {ganador[0]:+.1f})")

## Síntesis · Resultados observados

| Experimento | Fenómeno | Relación con el docking |
|---|---|---|
| 1 | Descenso estable hasta el mínimo local | **Minimización con MMFF94:** relajar la geometría de un ligando antes de acoplar |
| 2 | El tamaño de $\alpha$ afecta la convergencia del descenso | **`exhaustiveness`** regula el esfuerzo de búsqueda de poses en Vina; no es equivalente al tamaño de paso |
| 3 | Distintos puntos iniciales pueden alcanzar mínimos diferentes | **Búsqueda de poses:** procedimiento aproximado que genera y clasifica candidatos, sin garantía de mínimo global |

> **Distinción clave:** el descenso de gradiente ilustra una minimización local sobre un paisaje definido. El docking utiliza una función de puntuación y una búsqueda heurística sobre poses; ambos se relacionan con paisajes energéticos, pero el procedimiento y el objetivo computacional no son equivalentes.

## Cierre · Conexión con el *docking molecular*

1. **¿Por qué el docking da muchas poses?** Porque cada búsqueda desde una pose inicial distinta puede caer en un valle distinto del paisaje de unión. El programa clasifica las poses según su función de puntuación; las búsquedas heurísticas no garantizan el mínimo global.

2. **¿Por qué minimizar con MMFF94 no basta?** La minimización lleva al valle *más cercano*; no garantiza el mejor. Por eso después del docking el taller pasa a MM/GBSA y dinámica molecular: son formas de **auditar** que el valle elegido es realmente profundo y estable.

3. **¿Dónde estamos en el mapa del taller?**
   - **Actividad 1:** potencial → fuerza. 
   - **Actividad 2 (esta):** paisaje → descenso de gradiente → mínimo local vs. global. 
   - **Actividad 3:** modelo didáctico de términos de campo de fuerza.
   - **Actividades 4–5:** mover los átomos en el tiempo → dinámica molecular.

> **Pregunta guía del curso:** una hipótesis de unión "honesta" empieza por saber *cómo* se encontró el valle que la sostiene, y *cuánto* se exploró el paisaje alrededor.